# Notebook 02 — Data Cleaning and Staging Layer

**Project:** Enterprise E-Commerce Operations and Customer Experience Control Tower  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Phase:** 4 — Cleaning and Staging  

## Objectives
- Apply data cleaning transformations systematically across all 9 raw datasets
- Standardize data types, strings, and datetime columns
- Engineer data quality and operational anomaly flags (e.g. sequence errors, status mismatches)
- Aggregate multi-row tables (`payments`, `reviews`, `geolocation`) to prevent grain mismatch and double-counting during downstream joins
- Translate Portuguese product categories and engineer volume and size band features
- Save clean, standardized datasets into `data/staging/`

---
## 0. Setup & Dependencies

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_PATH = '../data/raw/'
STAGING_PATH = '../data/staging/'
os.makedirs(STAGING_PATH, exist_ok=True)

print('Data directories initialized.')

---
## 1. Cleaning Orders Table & Engineering Quality Flags

**Transformations:**
- Parse all 5 timestamp columns with error coercion
- Clean status text (lowercase, stripped)
- Quality Flags:
  - `missing_approval_flag`
  - `missing_carrier_flag`
  - `missing_delivery_flag`
  - `invalid_timestamp_sequence_flag` (e.g. delivery date before purchase date)
  - `delivered_status_mismatch_flag` (status = 'delivered' but customer delivery date is NULL)

In [ ]:
orders = pd.read_csv(os.path.join(RAW_PATH, 'olist_orders_dataset.csv'))
orders['order_status'] = orders['order_status'].str.strip().str.lower()

date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

# Create quality flags
orders['missing_approval_flag'] = orders['order_approved_at'].isnull().astype(int)
orders['missing_carrier_flag'] = orders['order_delivered_carrier_date'].isnull().astype(int)
orders['missing_delivery_flag'] = orders['order_delivered_customer_date'].isnull().astype(int)

seq_err = (
    (orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']) |
    (orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']) |
    (orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date'])
).fillna(False)
orders['invalid_timestamp_sequence_flag'] = seq_err.astype(int)

status_mismatch = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())
orders['delivered_status_mismatch_flag'] = status_mismatch.astype(int)

orders.to_csv(os.path.join(STAGING_PATH, 'stg_orders.csv'), index=False)
print(f"stg_orders: {orders.shape[0]:,} rows x {orders.shape[1]} cols")
orders.head(3)

---
## 2. Cleaning Order Items

**Transformations:**
- Parse shipping limit timestamp
- Validate numeric prices and freights
- Feature Engineering:
  - `item_total_value = price + freight_value`
  - `freight_to_price_ratio = freight_value / price`

In [ ]:
items = pd.read_csv(os.path.join(RAW_PATH, 'olist_order_items_dataset.csv'))
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')
items['price'] = pd.to_numeric(items['price'], errors='coerce').fillna(0)
items['freight_value'] = pd.to_numeric(items['freight_value'], errors='coerce').fillna(0).clip(lower=0)

items['item_total_value'] = items['price'] + items['freight_value']
items['freight_to_price_ratio'] = np.where(items['price'] > 0, items['freight_value'] / items['price'], 0.0)

items.to_csv(os.path.join(STAGING_PATH, 'stg_order_items.csv'), index=False)
print(f"stg_order_items: {items.shape[0]:,} rows x {items.shape[1]} cols")
items.head(3)

---
## 3. Cleaning Customers & Sellers

**Transformations:**
- Text standardization (city in Title Case, state in uppercase)
- Preservation of both `customer_id` and `customer_unique_id`

In [ ]:
customers = pd.read_csv(os.path.join(RAW_PATH, 'olist_customers_dataset.csv'))
customers['customer_city'] = customers['customer_city'].str.strip().str.title()
customers['customer_state'] = customers['customer_state'].str.strip().str.upper()
customers.to_csv(os.path.join(STAGING_PATH, 'stg_customers.csv'), index=False)

sellers = pd.read_csv(os.path.join(RAW_PATH, 'olist_sellers_dataset.csv'))
sellers['seller_city'] = sellers['seller_city'].str.strip().str.title()
sellers['seller_state'] = sellers['seller_state'].str.strip().str.upper()
sellers.to_csv(os.path.join(STAGING_PATH, 'stg_sellers.csv'), index=False)

print(f"stg_customers: {customers.shape[0]:,} rows, stg_sellers: {sellers.shape[0]:,} rows")

---
## 4. Products Table: Translation & Dimension Engineering

**Transformations:**
- Merge English category translations
- Fill missing category names with `'other'` / `'outros'`
- Compute `product_volume_cm3 = length * height * width`
- Engineer categorical `product_size_band` and `product_weight_band`

In [ ]:
products = pd.read_csv(os.path.join(RAW_PATH, 'olist_products_dataset.csv'))
cat_trans = pd.read_csv(os.path.join(RAW_PATH, 'product_category_name_translation.csv'))

products = products.merge(cat_trans, on='product_category_name', how='left')
products['product_category_name'] = products['product_category_name'].fillna('outros')
products['product_category_name_english'] = products['product_category_name_english'].fillna('other')

products.loc[products['product_category_name'] == 'pc_gamer', 'product_category_name_english'] = 'pc_gamer'
products.loc[products['product_category_name'] == 'portateis_cozinha_e_preparadores_de_alimentos', 'product_category_name_english'] = 'kitchen_small_appliances'

for c in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[c] = pd.to_numeric(products[c], errors='coerce')

products['product_volume_cm3'] = products['product_length_cm'] * products['product_height_cm'] * products['product_width_cm']

def categorize_volume(v):
    if pd.isna(v): return 'Unknown'
    elif v < 5000: return 'Small (<5L)'
    elif v < 20000: return 'Medium (5-20L)'
    elif v < 60000: return 'Large (20-60L)'
    else: return 'Extra Large (>=60L)'

def categorize_weight(w):
    if pd.isna(w): return 'Unknown'
    elif w < 1000: return 'Light (<1kg)'
    elif w < 5000: return 'Medium (1-5kg)'
    else: return 'Heavy (>=5kg)'

products['product_size_band'] = products['product_volume_cm3'].apply(categorize_volume)
products['product_weight_band'] = products['product_weight_g'].apply(categorize_weight)

products.to_csv(os.path.join(STAGING_PATH, 'stg_products.csv'), index=False)
print(f"stg_products: {products.shape[0]:,} rows x {products.shape[1]} cols")
products.head(3)

---
## 5. Payments: Detailed & Order-Level Staging

**Why Order-Level Aggregation is Critical:**
- 2,961 orders contain multiple payment splits (e.g. vouchers + credit card)
- Aggregating at the order level creates a 1:1 relationship with orders, completely eliminating row multiplication in fact tables

In [ ]:
payments = pd.read_csv(os.path.join(RAW_PATH, 'olist_order_payments_dataset.csv'))
payments['payment_type'] = payments['payment_type'].str.strip().str.lower()
payments.loc[payments['payment_type'] == 'not_defined', 'payment_type'] = 'unknown'
payments['payment_value'] = pd.to_numeric(payments['payment_value'], errors='coerce').fillna(0).clip(lower=0)
payments['payment_installments'] = pd.to_numeric(payments['payment_installments'], errors='coerce').fillna(1).astype(int)

# Save line-item level payments
payments.to_csv(os.path.join(STAGING_PATH, 'stg_order_payments.csv'), index=False)

# Primary payment type: highest value payment for each order
pay_sorted = payments.sort_values(['order_id', 'payment_value'], ascending=[True, False])
primary_pay = pay_sorted.groupby('order_id').first()[['payment_type']].rename(columns={'payment_type': 'primary_payment_type'})

pay_agg = payments.groupby('order_id').agg(
    payment_value_total=('payment_value', 'sum'),
    payment_installments_max=('payment_installments', 'max'),
    payment_record_count=('payment_sequential', 'count')
).reset_index()

pay_agg = pay_agg.merge(primary_pay, on='order_id', how='left')
pay_agg['multi_payment_flag'] = (pay_agg['payment_record_count'] > 1).astype(int)

pay_agg.to_csv(os.path.join(STAGING_PATH, 'stg_payments_order_agg.csv'), index=False)
print(f"stg_payments_order_agg: {pay_agg.shape[0]:,} orders (Aggregated)")
pay_agg.head(3)

---
## 6. Reviews: Detailed & Order-Level Staging

**Transformations:**
- Parse creation and answer dates
- Deduplicate `review_id`
- Compute `review_response_hours`
- Aggregate to order-level (average score, latest score, response hours, comment flag)

In [ ]:
reviews = pd.read_csv(os.path.join(RAW_PATH, 'olist_order_reviews_dataset.csv'))
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

reviews['review_response_hours'] = (
    (reviews['review_answer_timestamp'] - reviews['review_creation_date']).dt.total_seconds() / 3600.0
).clip(lower=0)

reviews_clean = reviews.sort_values('review_answer_timestamp', ascending=False).drop_duplicates(subset=['review_id'])
reviews_clean.to_csv(os.path.join(STAGING_PATH, 'stg_order_reviews.csv'), index=False)

reviews['has_comment_flag'] = (reviews['review_comment_message'].notna() | reviews['review_comment_title'].notna()).astype(int)

rev_agg = reviews.groupby('order_id').agg(
    review_score_avg=('review_score', 'mean'),
    review_score_latest=('review_score', 'last'),
    review_count=('review_id', 'count'),
    review_comment_flag=('has_comment_flag', 'max'),
    review_response_hours=('review_response_hours', 'mean')
).reset_index()

rev_agg['review_score_avg'] = rev_agg['review_score_avg'].round(2)
rev_agg['review_response_hours'] = rev_agg['review_response_hours'].round(1)

rev_agg.to_csv(os.path.join(STAGING_PATH, 'stg_reviews_order_agg.csv'), index=False)
print(f"stg_reviews_order_agg: {rev_agg.shape[0]:,} orders (Aggregated)")
rev_agg.head(3)

---
## 7. Geolocation: Outlier Filtering & ZIP Prefix Aggregation

**Transformations:**
- Drop exact row duplicates
- Filter out GPS coordinates outside Brazil's bounding box
- Aggregate coordinates to unique `geolocation_zip_code_prefix` using the median coordinate

In [ ]:
geo = pd.read_csv(os.path.join(RAW_PATH, 'olist_geolocation_dataset.csv'))
geo_dedup = geo.drop_duplicates()

valid_coords = (
    (geo_dedup['geolocation_lat'] >= -35.0) & (geo_dedup['geolocation_lat'] <= 6.0) & 
    (geo_dedup['geolocation_lng'] >= -75.0) & (geo_dedup['geolocation_lng'] <= -33.0)
)
geo_clean = geo_dedup[valid_coords].copy()
geo_clean['geolocation_city'] = geo_clean['geolocation_city'].str.strip().str.title()
geo_clean['geolocation_state'] = geo_clean['geolocation_state'].str.strip().str.upper()

geo_zip = (
    geo_clean.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        latitude=('geolocation_lat', 'median'),
        longitude=('geolocation_lng', 'median'),
        city=('geolocation_city', 'first'),
        state=('geolocation_state', 'first')
    )
)

geo_zip.to_csv(os.path.join(STAGING_PATH, 'stg_geolocation_zip.csv'), index=False)
print(f"stg_geolocation_zip: {geo_zip.shape[0]:,} unique ZIP codes (100% Unique PK)")
geo_zip.head(3)

---
## 8. Summary of Staged Artifacts

In [ ]:
staged_files = [f for f in os.listdir(STAGING_PATH) if f.endswith('.csv')]
summary = []
for f in staged_files:
    df_temp = pd.read_csv(os.path.join(STAGING_PATH, f))
    summary.append({
        'File': f,
        'Rows': f"{df_temp.shape[0]:,}",
        'Cols': df_temp.shape[1]
    })
pd.DataFrame(summary)